# PyMolFit 02: Building a Na D-region fit

This notebook adds one modelling decision at a time: a selected fit interval, instrumental broadening, wavelength alignment, astrophysical masks, and a composite line-spread function. Every step uses the same HARPS pixels and seasonal MIPAS/GDAS atmosphere.

The comparison statistic is the normalized data-minus-model RMS outside the broad Na D masks. It is a useful internal diagnostic, but it is not the unknown true telluric-correction error.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pymolfit import correct_file

candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next((p.resolve() for p in candidates if (p / "data").is_dir()), None)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not locate tutorials/data")
INPUT = TUTORIAL_ROOT / "data" / "harps_nad_crop_air.fits"
OUTPUT = TUTORIAL_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
PLOT_RANGE = (5887.5, 5898.8)
FIT_RANGE = ((0.58825, 0.59065),)
NA_MASKS_MICRON = ((0.58888, 0.58912), (0.58948, 0.58978))
NA_MASKS_ANGSTROM = ((5888.8, 5891.2), (5894.8, 5897.8))

## Shared plotting and comparison helpers

The upper panel always shows the observed and current corrected spectra. From Step 2 onward the lower panel shows the point-by-point change from the previous correction. Red bands identify astrophysical pixels that the final fits do not use.

In [ ]:
def validation_rms(result):
    wavelength = result.spectrum.to_air().to_unit("angstrom").wavelength
    continuum = np.asarray(result.continuum, dtype=float)
    residual = np.asarray(result.spectrum.flux - result.model_flux, dtype=float)
    valid = (
        np.isfinite(residual)
        & np.isfinite(continuum)
        & (continuum != 0)
        & (wavelength >= PLOT_RANGE[0])
        & (wavelength <= PLOT_RANGE[1])
    )
    for lower, upper in NA_MASKS_ANGSTROM:
        valid &= ~((wavelength >= lower) & (wavelength <= upper))
    normalized = residual[valid] / continuum[valid]
    return float(np.sqrt(np.mean(normalized**2)))


def plot_step(result, title, color, previous=None):
    observed = result.spectrum.to_air().to_unit("angstrom")
    corrected = result.corrected.to_air().to_unit("angstrom")
    visible = (observed.wavelength >= PLOT_RANGE[0]) & (observed.wavelength <= PLOT_RANGE[1])
    scale = np.nanmedian(observed.flux[visible])

    if previous is None:
        figure, main = plt.subplots(figsize=(11, 4))
        change = None
    else:
        figure, (main, change) = plt.subplots(
            2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": (3, 1)}
        )

    main.plot(observed.wavelength, observed.flux / scale, color="black", alpha=0.60, label="Observed")
    main.plot(corrected.wavelength, corrected.flux / scale, color=color, label="Corrected")
    main.set_ylabel("Flux / local median")
    main.set_title(title)
    main.legend()

    for lower, upper in NA_MASKS_ANGSTROM:
        main.axvspan(lower, upper, color="tab:red", alpha=0.10)

    if previous is None:
        main.set_xlabel("Air wavelength [Angstrom]")
    else:
        old = previous.corrected.to_air().to_unit("angstrom")
        old_flux = np.interp(corrected.wavelength, old.wavelength, old.flux)
        relative = np.full(corrected.flux.shape, np.nan, dtype=float)
        valid = np.isfinite(corrected.flux) & np.isfinite(old_flux) & (old_flux != 0)
        relative[valid] = 100 * (corrected.flux[valid] / old_flux[valid] - 1)
        change.plot(corrected.wavelength, relative, color=color, linewidth=0.9)
        change.axhline(0, color="black", linewidth=0.8)
        change.set_ylabel("Change [%]")
        change.set_xlabel("Air wavelength [Angstrom]")
        change.text(
            0.01, 0.90,
            f"Normalized residual RMS: {validation_rms(previous):.4f} -> {validation_rms(result):.4f}",
            transform=change.transAxes, va="top",
        )
        for lower, upper in NA_MASKS_ANGSTROM:
            change.axvspan(lower, upper, color="tab:red", alpha=0.10)

    main.set_xlim(*PLOT_RANGE)
    figure.tight_layout()
    plt.show()

## Step 1: Selected wavelength interval

The first fit uses only the predetermined Na D neighbourhood, but it still has a Gaussian-free, perfectly sampled instrumental model and it allows the astrophysical Na lines to affect the continuum. It is deliberately incomplete and establishes the baseline.

In [ ]:
result_1 = correct_file(
    input_path=INPUT,                         # Reduced one-dimensional FITS spectrum
    wavelength_medium="air",               # Describe the input wavelength convention
    aer_catalog="auto",                   # Managed AER 3.9 molecular catalogue
    gdas_mode="average",                  # Deterministic seasonal atmosphere
    solve_continuum_linear=True,             # Stable continuum solution for large flux values
    fit_ranges=FIT_RANGE,                    # Only this interval constrains the fit
)
plot_step(result_1, "Step 1: selected interval only", "tab:blue")

## Step 2: Gaussian instrumental broadening

Atmospheric lines are calculated at high resolution and must be convolved with the spectrograph line-spread function. `2.17` pixels is a HARPS-informed starting value, not a forced answer; the optimizer refines it within the stated bounds.

In [ ]:
result_2 = correct_file(
    input_path=INPUT,
    wavelength_medium="air",
    aer_catalog="auto",
    gdas_mode="average",
    solve_continuum_linear=True,
    fit_ranges=FIT_RANGE,
    lsf_sigma_pixels=2.17,                   # Initial Gaussian sigma in detector pixels
    fit_lsf_sigma=True,                     # Fit the Gaussian width
    lsf_sigma_bounds=(0.5, 4.0),            # Prevent a degenerate or implausibly broad kernel
)
plot_step(result_2, "Step 2: Gaussian instrumental broadening", "tab:orange", result_1)

## Step 3: Wavelength alignment

A small wavelength mismatch leaves adjacent positive and negative residuals. The allowed shift is narrow enough to prevent the model from matching unrelated stellar features.

In [ ]:
result_3 = correct_file(
    input_path=INPUT,
    wavelength_medium="air",
    aer_catalog="auto",
    gdas_mode="average",
    solve_continuum_linear=True,
    fit_ranges=FIT_RANGE,
    lsf_sigma_pixels=2.17,
    fit_lsf_sigma=True,
    lsf_sigma_bounds=(0.5, 4.0),
    fit_wavelength_shift=True,              # Fit a constant model displacement
    wavelength_shift_bounds=(-2e-5, 2e-5),  # +/-0.2 Angstrom because values are in micron
)
plot_step(result_3, "Step 3: broadening and wavelength alignment", "tab:green", result_2)

## Step 4: Protect astrophysical Na D absorption

PyMolFit does not know that the broad stellar Na D troughs and narrow circumstellar cores are astrophysical. Both complete Na regions are therefore excluded while atmospheric and continuum parameters are estimated. They remain in the output and still receive the fitted telluric correction.

In [ ]:
result_4 = correct_file(
    input_path=INPUT,
    wavelength_medium="air",
    aer_catalog="auto",
    gdas_mode="average",
    solve_continuum_linear=True,
    fit_ranges=FIT_RANGE,
    exclude_ranges=NA_MASKS_MICRON,         # Ignore the broad Na D regions during fitting
    lsf_sigma_pixels=2.17,
    fit_lsf_sigma=True,
    lsf_sigma_bounds=(0.5, 4.0),
    fit_wavelength_shift=True,
    wavelength_shift_bounds=(-2e-5, 2e-5),
)
plot_step(result_4, "Step 4: astrophysical Na D regions masked", "tab:red", result_3)

## Step 5: Composite LSF and curved continuum

The final model includes detector-pixel integration, Gaussian core broadening, Lorentzian wings, and quadratic continuum curvature. The fitted atmospheric scale is then informed by telluric pixels rather than by the stellar Na D shape. No fitted value from Molecfit is inserted into this configuration.

In [ ]:
result_final = correct_file(
    input_path=INPUT,
    output_path=OUTPUT / "02_nad_corrected.ecsv",
    product_path=OUTPUT / "02_nad_fit_product.ecsv",
    wavelength_medium="air",
    aer_catalog="auto",
    gdas_mode="average",
    continuum_order=2,                     # Quadratic local source/throughput shape
    solve_continuum_linear=True,
    fit_ranges=FIT_RANGE,
    exclude_ranges=NA_MASKS_MICRON,
    lsf_box_width_pixels=1.0,              # Detector-pixel integration
    lsf_sigma_pixels=2.17,
    fit_lsf_sigma=True,
    lsf_sigma_bounds=(0.5, 4.0),
    lsf_lorentz_fwhm_pixels=0.5,           # Initial non-Gaussian wing width
    fit_lsf_lorentz_fwhm=True,
    lsf_lorentz_fwhm_bounds=(0.0, 6.0),
    fit_wavelength_shift=True,
    wavelength_shift_bounds=(-2e-5, 2e-5),
)
plot_step(result_final, "Step 5: composite LSF and quadratic continuum", "tab:purple", result_4)

## Step 6: Final scientific diagnostics

The continuum is a nuisance model, not a physical model of the stellar Na D lines. Inspect the unmasked data-model alignment, atmospheric transmission, corrected spectrum, residual structure, and parameter bounds together.

In [ ]:
wave = result_final.spectrum.to_air().to_unit("angstrom").wavelength
flux = np.asarray(result_final.spectrum.flux, dtype=float)
model = np.asarray(result_final.model_flux, dtype=float)
continuum = np.asarray(result_final.continuum, dtype=float)
corrected = np.asarray(result_final.corrected.flux, dtype=float)
transmission = np.asarray(result_final.transmission, dtype=float)
visible = (wave >= PLOT_RANGE[0]) & (wave <= PLOT_RANGE[1])
scale = np.nanmedian(flux[visible])
residual = np.divide(flux - model, continuum, out=np.full_like(flux, np.nan), where=continuum != 0)

figure, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(wave, flux / scale, color="black", label="Observed")
axes[0].plot(wave, model / scale, color="tab:orange", label="Model")
axes[0].set_ylabel("Normalized flux")
axes[0].legend()
axes[1].plot(wave, transmission, color="tab:blue")
axes[1].set_ylabel("Transmission")
axes[2].plot(wave, corrected / scale, color="tab:green", label="Corrected")
axes[2].plot(wave, continuum / scale, color="0.4", linestyle="--", label="Continuum")
axes[2].set_ylabel("Normalized flux")
axes[2].legend()
axes[3].plot(wave, residual, color="tab:purple")
axes[3].axhline(0, color="black", linewidth=0.8)
axes[3].set_ylabel("Residual / continuum")
axes[3].set_xlabel("Air wavelength [Angstrom]")
for axis in axes:
    axis.set_xlim(*PLOT_RANGE)
    axis.grid(alpha=0.15)
    for lower, upper in NA_MASKS_ANGSTROM:
        axis.axvspan(lower, upper, color="tab:red", alpha=0.10)
figure.tight_layout()
plt.show()

print("Success:", result_final.success)
print("Species scales:", result_final.species_scales)
print("Wavelength shift [Angstrom]:", result_final.wavelength_shift * 1e4)
print("LSF (Gaussian sigma, box, Lorentz FWHM) [pixels]:", (
    result_final.lsf_sigma_pixels, result_final.lsf_box_width_pixels, result_final.lsf_lorentz_fwhm_pixels
))
print("Parameters at bounds:", result_final.parameter_bound_status or "none")
print("Fit pixels:", np.count_nonzero(result_final.fit_mask))

## Conclusions

The final configuration is specialized for this Na D validation window. The managed catalogue still contains the standard atmospheric species, but only molecules with measurable absorption in the selected unmasked pixels can be constrained. A fit in this narrow window must not be described as validating the correction over an unrelated wavelength range. Tutorial 03 extends the workflow to a broad spectrum.